In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pickagm.distributions import ensemble_ks_bounds

from phd_project.config.config import load_config 
from phd_project.scripts.cache_utils import fingerprint, load_or_compute
from phd_project.scripts.WP1_ground_motion_set.gm_selection import (
    calculate_gcim_distributions_for_sites,
    _selection_ctx_fingerprint_inputs,
)

from phd_project.scripts.WP1_ground_motion_set.setup_AvgSA06_gm_selection import setup_AvgSA06_gcim_gm_selection

cfg = load_config()

C:\Users\clemettn\Documents\phd\phd_project\scripts\WP1_ground_motion_set\gm_selection.py:15: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


# 034 — GCIM distributions, AvgSA([0, 6])

Computes the GCIM (Generalised Conditional Intensity Measure) target distributions for every
`(site, iml)` disaggregation, conditioned on **AvgSA([0, 6])**. For each site/intensity level the
GMMs and correlation models are run over that site's disaggregation to produce the target
`stats` / `pdfs` / `cdfs`. This is the slow target-building step that precedes record selection.

**Upstream** — read by `setup_AvgSA06_gcim_gm_selection()` (`setup_AvgSA06_gm_selection.py`):

| Input | Config key |
|---|---|
| IML-based disaggregations (nb 021) | `cfg["proc_data"]["AvgSA_06_disagg_data_gm_selection"]` |
| Per-(site, imt, iml) poe stats | `cfg["proc_data"]["AvgSA_06_disagg_stats_gm_selection"]` |
| Per-site IML subset (one MSA stripe each) | `cfg["proc_data"]["AvgSA_06_imls_for_selection"]` |
| Site model | `cfg["hazard_models"]["eshm20_wp1_site_model"]` |
| Median-branch GMM logic tree | `cfg["hazard_models"]["eshm20_AvgSA_06_median_lt"]` |
| Correlation flatfiles + GM database | `cfg["proc_data"]["corr_model"]`, `cfg["proc_data"]["gm_database"]` (db used for selection, not the gcim calc) |

**Downstream** — `gcim_dist_AvgSA_06.pickle` is consumed by
`035-gm_selection_AvgSA_06_stage1_compute.ipynb`, which fingerprints it as `gcim_file`, so the
manifest written here closes the provenance chain into Stage-1.

**Output** — `cfg["proc_data"]["gcim_dists"] / "gcim_dist_AvgSA_06.pickle"`, plus a
`gcim_dist_AvgSA_06.pickle.manifest.json` provenance sidecar recording a content hash of every
input (upstream files, selection context, percentiles, `pickagm` version) along with the git
commit and timestamp.

**Run order** — run top to bottom. The compute cell is provenance-cached via
`cache_utils.load_or_compute`: a re-run with unchanged inputs reloads the pickle instantly
(`[cache] ... loaded (inputs match).`) instead of recomputing. Set `FORCE_RECOMPUTE = True` to
rebuild and overwrite regardless of the cache.

In [ ]:
# set up the gcim / record selection
site_iml_disaggs, disagg_stats, site_model, basic_selection_ctx, _ = setup_AvgSA06_gcim_gm_selection()
percentiles = [0.05, 0.16, 0.33, 0.5, 0.67, 0.84, 0.95]

In [ ]:
# NOTE: set True for the one-time rebuild after the (site, poe) -> (site, iml)
# re-keying. The provenance fingerprint hashes input files/params, NOT the dict-key
# scheme, so a code-only re-key is invisible to the cache: without forcing, the old
# (site, poe)-keyed pickle would be reloaded. Set back to False once rebuilt.
FORCE_RECOMPUTE = True   # True to rebuild + overwrite, ignoring the cache

gcim_dist_fp = cfg["proc_data"]["gcim_dists"] / "gcim_dist_AvgSA_06.pickle"

# Inputs that determine the gcim distributions, hashed for the provenance manifest.
# Upstream files are hashed by their bytes (cheap, stable); the selection context and
# percentiles are hashed by value. Keep these keys in sync with the files read by
# setup_AvgSA06_gcim_gm_selection().
gcim_input_fp = fingerprint(
    disagg_data_file=cfg["proc_data"]["AvgSA_06_disagg_data_gm_selection"],
    disagg_stats_file=cfg["proc_data"]["AvgSA_06_disagg_stats_gm_selection"],
    imls_for_selection_file=cfg["proc_data"]["AvgSA_06_imls_for_selection"],
    site_model_file=cfg["hazard_models"]["eshm20_wp1_site_model"],
    gmm_lt_file=cfg["hazard_models"]["eshm20_AvgSA_06_median_lt"],
    percentiles=tuple(percentiles),
    **_selection_ctx_fingerprint_inputs(basic_selection_ctx),
)

In [ ]:
# Compute the gcim distributions (or load the cached pickle if inputs are unchanged).
# load_or_compute writes gcim_dist_AvgSA_06.pickle + its .manifest.json provenance sidecar.
gcim_dists = load_or_compute(
    gcim_dist_fp,
    gcim_input_fp,
    lambda: calculate_gcim_distributions_for_sites(
        site_iml_disaggs, disagg_stats, site_model, basic_selection_ctx, percentiles,
    ),
    force_recompute=FORCE_RECOMPUTE,
)